# 金融数据清洗与训练闭环（SFT → RM → DPO/PPO）

本 Notebook 给出一条“可审计”的落地路径：
1. 按 MedicalGPT `docs/datasets.md` 规范明确输出格式；
2. 明确说明为什么筛、如何控噪、如何对齐行业研究报告目标；
3. 产出 SFT/RM/RL + Eval 数据，并提供训练命令。


## 数据清洗原则（结合 MedicalGPT + 金融场景）

- **格式先行**：先把原始样本映射成统一字段，再做质量控制；
- **规则可解释**：长度、拒答、重复、夸大收益表达都做显式规则；
- **任务对齐**：保留“结论-驱动-估值-风险-跟踪指标”的报告结构；
- **偏好可追溯**：chosen/rejected 有明确构造逻辑；
- **评测可复现**：保留分层评测集与清洗报告。


In [ ]:
# 可选：安装依赖
# !pip install -U datasets pandas numpy scikit-learn


In [ ]:
from pathlib import Path
import json
import subprocess

SOURCE = "BAAI/IndustryInstruction_Finance-Economics"  # 或本地路径 .jsonl/.json/.parquet
OUT_DIR = Path("data/finance")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MIN_QUALITY = 0.58


## 运行清洗脚本

脚本会完成：
- 字段标准化（question/answer）；
- 规则过滤（空样本、长度、重复、低质量、风险表达）；
- 数据增强（行业研究报告模板注入）；
- 偏好数据构造（response_chosen/rejected）；
- 导出 MedicalGPT 所需 JSONL 与 `cleaning_report.json`。


In [ ]:
cmd = [
    "python", "finance_data_cleaning.py",
    "--source", SOURCE,
    "--output_dir", str(OUT_DIR),
    "--min_quality", str(MIN_QUALITY),
]
print(" ".join(cmd))

# 如网络受限，可先手动下载数据集到本地再把 SOURCE 改为本地文件路径
# subprocess.run(cmd, check=True)


In [ ]:
report_path = OUT_DIR / "cleaning_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print("请先运行上一格命令生成 cleaning_report.json")


## 输出文件说明

- `finance_sft_train.jsonl` / `finance_sft_val.jsonl`：SFT
- `finance_rm_train.jsonl` / `finance_rm_val.jsonl`：RM/DPO
- `finance_rl_train.jsonl`：PPO/GRPO
- `finance_eval.jsonl`：评测集（离线评测/人工评审）


In [ ]:
for p in [
    "finance_sft_train.jsonl",
    "finance_sft_val.jsonl",
    "finance_rm_train.jsonl",
    "finance_rm_val.jsonl",
    "finance_rl_train.jsonl",
    "finance_eval.jsonl",
]:
    fp = OUT_DIR / p
    print(p, "exists" if fp.exists() else "missing")


## 训练命令模板（Qwen2.5-7B）


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-7B"

SFT_CMD = f"""python supervised_finetuning.py \\
  --model_name_or_path {BASE_MODEL} \\
  --train_file_dir data/finance \\
  --validation_file_dir data/finance \\
  --do_train --do_eval \\
  --use_peft True --lora_rank 8 --lora_alpha 16 --lora_dropout 0.05 \\
  --num_train_epochs 2 --learning_rate 2e-5 \\
  --output_dir outputs-finance-sft"""

RM_CMD = """python reward_modeling.py \\
  --model_name_or_path outputs-finance-sft \\
  --train_file_dir data/finance \\
  --validation_file_dir data/finance \\
  --do_train --do_eval \\
  --num_train_epochs 1 --learning_rate 1e-5 \\
  --output_dir outputs-finance-rm"""

DPO_CMD = """python dpo_training.py \\
  --model_name_or_path outputs-finance-sft \\
  --train_file_dir data/finance \\
  --validation_file_dir data/finance \\
  --beta 0.1 --num_train_epochs 1 --learning_rate 1e-6 \\
  --output_dir outputs-finance-dpo"""

PPO_CMD = """python ppo_training.py \\
  --sft_model_name_or_path outputs-finance-sft \\
  --reward_model_name_or_path outputs-finance-rm \\
  --train_file_dir data/finance \\
  --output_dir outputs-finance-ppo"""

for title, cmd in [("SFT", SFT_CMD), ("RM", RM_CMD), ("DPO", DPO_CMD), ("PPO", PPO_CMD)]:
    print(f"===== {title} =====")
    print(cmd)
    print()


## 评测集建议（证据链）

- 覆盖行业：金融/地产/科技/消费等；
- 覆盖能力：摘要、归因、估值、风险提示、反例分析；
- 记录维度：difficulty、quality_score、source；
- 评估方式：自动指标 + 分析师双盲偏好打分。
